# Week 2 Day 4 — Tools (Anthropic edition)

> **Targets Gradio 6.** No `type="messages"`; history flattened via `to_text()`.

## Airline AI Assistant with tool use

OpenAI vs Anthropic tool format — the differences that matter:

| | OpenAI | Anthropic |
|---|---|---|
| Tool definition | `{"type":"function","function":{...,"parameters":{...}}}` | `{"name","description","input_schema":{...}}` |
| "Wants a tool" signal | `finish_reason == "tool_calls"` | `stop_reason == "tool_use"` |
| Where the call is | `message.tool_calls[0].function` | a block in `response.content` with `type=="tool_use"` |
| Arguments | `.function.arguments` — a JSON **string** (`json.loads` it) | `block.input` — **already a dict** |
| Result goes back as | a `{"role":"tool","tool_call_id":...}` message | a **user** message with a `{"type":"tool_result","tool_use_id":...}` block |

In [2]:
import os
import json
from dotenv import load_dotenv
import anthropic
import gradio as gr

load_dotenv(override=True)
print("Anthropic key set" if os.getenv("ANTHROPIC_API_KEY") else "Anthropic key NOT set")

MODEL = "claude-haiku-4-5"
client = anthropic.Anthropic()

Anthropic key set


In [3]:
def to_text(content):
    """Gradio 6 gives message content as a list of blocks; older Gradio gave a plain string.
    Anthropic wants a plain string here, so flatten it. Handles both versions safely."""
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get("text", "")
    if isinstance(content, list):
        return "".join(b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text")
    return str(content)

In [4]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

## Baseline chatbot (no tools yet)

In [5]:
def chat(message, history):
    messages = [{"role": h["role"], "content": to_text(h["content"])} for h in history]
    messages.append({"role": "user", "content": message})
    response = client.messages.create(model=MODEL, max_tokens=500, system=system_message, messages=messages)
    return response.content[0].text

gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Step 1: a function for the model to call

In [6]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"

get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

## Step 2: describe the tool — Anthropic format (`input_schema`, no wrapper)

In [7]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "input_schema": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
    },
}
tools = [price_function]
tools

[{'name': 'get_ticket_price',
  'description': 'Get the price of a return ticket to the destination city.',
  'input_schema': {'type': 'object',
   'properties': {'destination_city': {'type': 'string',
     'description': 'The city that the customer wants to travel to'}},
   'required': ['destination_city']}}]

## Step 3: let Claude call the tool

In [8]:
def text_from(response):
    return "".join(b.text for b in response.content if b.type == "text")

def handle_tool_calls(response):
    results = []
    for block in response.content:
        if block.type == "tool_use":
            if block.name == "get_ticket_price":
                result = get_ticket_price(block.input.get("destination_city"))
            elif block.name == "set_ticket_price":
                set_ticket_price(block.input.get("destination_city"), block.input.get("price"))
                result = f"Price for {block.input.get('destination_city')} set to ${block.input.get('price')}"
            else:
                result = "Unknown tool"
            results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": result,
            })
    return results

In [ ]:
def chat(message, history):
    messages = [{"role": h["role"], "content": to_text(h["content"])} for h in history]
    messages.append({"role": "user", "content": message})
    response = client.messages.create(model=MODEL, max_tokens=500, system=system_message, messages=messages, tools=tools)

    while response.stop_reason == "tool_use":
        tool_results = handle_tool_calls(response)
        messages.append({"role": "assistant", "content": response.content})   # echo model's request
        messages.append({"role": "user", "content": tool_results})            # results as a USER turn
        response = client.messages.create(model=MODEL, max_tokens=500, system=system_message, messages=messages, tools=tools)

    return text_from(response)

gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 

Multiple tool calls (in one response, or chained) are handled for free: they're just multiple `tool_use` blocks, and the `while` loop plus `handle_tool_calls` already covers both.

## Step 4: back the tool with a database

In [10]:
import sqlite3
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    conn.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        row = conn.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),)).fetchone()
        return f"Ticket price to {city} is ${row[0]}" if row else "No price data available for this city"

def set_ticket_price(city, price):
    print(f"DATABASE TOOL: setting {city} = {price}", flush=True)   # see what actually arrives
    with sqlite3.connect(DB) as conn:
        conn.execute(
            'INSERT OR REPLACE INTO prices (city, price) VALUES (?, ?)',
            (city.lower(), price),
        )
        conn.commit()

In [11]:
with sqlite3.connect(DB) as conn:
    conn.execute('DELETE FROM prices WHERE price IS NULL')
    conn.commit()

In [12]:
for city, price in {"london": 799, "paris": 899, "tokyo": 1420, "sydney": 2999,"mumbai":1999}.items():
    set_ticket_price(city, price)
get_ticket_price("Mumbai")

DATABASE TOOL: setting london = 799
DATABASE TOOL: setting paris = 899
DATABASE TOOL: setting tokyo = 1420
DATABASE TOOL: setting sydney = 2999
DATABASE TOOL: setting mumbai = 1999
DATABASE TOOL CALLED: Getting price for Mumbai


'Ticket price to Mumbai is $1999.0'

In [ ]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 

## Exercise

Add a `set_ticket_price` tool: write its schema, add it to `tools`, and add an `elif block.name == "set_ticket_price"` branch in `handle_tool_calls`. The loop already supports any number of tools.

In [17]:
set_price_function ={
    "name": "set_ticket_price",
    "description" : "Set the price of the ticket of the destination city.",
    "input_schema" : {
        "type" : "object",
        "properties" : {
            "destination_city" :{
                "type": "string",
                "description": "The city that the customer wants to travel to."
            
            },
            "price" : {
                "type" : "integer",
                "description" : "The amount that needs to be paid to travel to the destination."
            },
        },
        "required": ["destination_city"],
    },
}

tools = [price_function, set_price_function]

In [19]:
print(json.dumps(tools, indent=2))

[
  {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "input_schema": {
      "type": "object",
      "properties": {
        "destination_city": {
          "type": "string",
          "description": "The city that the customer wants to travel to"
        }
      },
      "required": [
        "destination_city"
      ]
    }
  },
  {
    "name": "set_ticket_price",
    "description": "Set the price of the ticket of the destination city.",
    "input_schema": {
      "type": "object",
      "properties": {
        "destination_city": {
          "type": "string",
          "description": "The city that the customer wants to travel to."
        },
        "price": {
          "type": "integer",
          "description": "The amount that needs to be paid to travel to the destination."
        }
      },
      "required": [
        "destination_city"
      ]
    }
  }
]


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


DATABASE TOOL: setting mumbai = 1600


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


In [15]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


DATABASE TOOL: setting mumbai = None


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
